In [100]:
import os
import shutil
import re

In [101]:
# Function get all sub-directory under root directory
def get_all_subdirectories_under_one_level(root_dir: str):
    """
    Get all subdirectories directly under the given root directory (one level only).
    Return a list of full paths.
    """
    return [
        os.path.join(root_dir, name)
        if not os.path.isabs(name) else name
        for name in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, name))
    ]
def get_app_package_name(path: str) -> str:
    """
    Return the last directory name from a full path.
    Example:
        E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote
        -> 'adarshurs.android.vlcmobileremote'
    """
    return os.path.basename(os.path.normpath(path))
def create_directory_if_not_exist(path: str):
    """Tạo thư mục nếu chưa tồn tại."""
    if not os.path.exists(path):
        os.makedirs(path)

In [102]:
root_directory_source_code = r"E:\wearable-project-decompile-source\source-code-standalone"
root_directory_extract_code = r"E:\wearable-2-sharing-compliance-code-extraction"

In [103]:
array_sensitive_keyword = [
    {
        "category": "Device or other IDs",
        "keywords": [
            "IMEI", "MEID", "ESN", "deviceId", "android_id", "Settings.Secure.ANDROID_ID",
            "Build.SERIAL", "getSerial", "getDeviceId", "getImei", "getMeid",
            "BluetoothAdapter.getAddress", "WifiInfo.getMacAddress",
            "AdvertisingIdClient", "AdvertisingId", "AdId", "AD_ID",
            "Widevine", "WidevineDeviceId", "FirebaseInstallations", "FirebaseInstanceId",
            "InstanceId", "getToken", "IP Address", "InetAddress"
        ]
    },
    {
        "category": "Personal info",
        "keywords": [
            "name", "username", "user_name", "fullname", "full_name", "email", "emailAddress",
            "password", "pwd", "userId", "user_id", "accountId", "birthday", "birth_date",
            "dob", "gender", "sex", "phone", "phoneNumber", "mobile", "contact_number"
        ]
    },
    {
        "category": "Location",
        "keywords": [
            "LocationManager", "getLatitude", "getLongitude", "getAltitude",
            "location", "GPS", "FusedLocationProviderClient", "Geocoder",
            "LocationRequest", "LocationListener", "latitude", "longitude", "altitude"
        ]
    },
    {
        "category": "Health & fitness",
        "keywords": [
            "heartRate", "heart_rate", "stepCount", "steps", "calories", "sleep",
            "distance", "exercise", "workout", "activity", "bloodPressure", "bp",
            "oxygen", "spo2", "weight", "height", "medical", "fitness"
        ]
    },
    {
        "category": "App info & performance",
        "keywords": [
            "crash", "Crashlytics", "appLog", "logEvent", "CPU", "RAM", "MemoryInfo",
            "battery", "PerformanceMonitor", "onLowMemory", "getMemoryClass", "Logcat"
        ]
    },
    {
        "category": "Messages",
        "keywords": [
            "SmsManager", "sendTextMessage", "MMS", "email", "inbox", "chat", "messaging",
            "message", "subject", "sender", "recipient", "body", "compose", "sendMail"
        ]
    },
    {
        "category": "Financial info",
        "keywords": [
            "creditCard", "card_number", "bankAccount", "account_number", "iban",
            "bic", "transaction", "balance", "cvv", "cvc", "paypal", "stripe",
            "payment", "billing", "accountInfo", "finance"
        ]
    },
    {
        "category": "Photos or video",
        "keywords": [
            "photo", "video", "camera", "ImageCapture", "MediaStore.Images", "MediaStore.Video",
            "bitmap", "jpg", "png", "recordVideo", "takePicture", "Camera2", "FileOutputStream"
        ]
    },
    {
        "category": "Audio files",
        "keywords": [
            "AudioRecord", "MediaRecorder", "recordAudio", "audio", "microphone",
            "sound", "voice", "AAC", "MP3", "WAV"
        ]
    },
    {
        "category": "Files and docs",
        "keywords": [
            "FileOutputStream", "FileInputStream", "DocumentFile", "Storage", "recording",
            "voiceNote", "userMusic", "audioNote", "recordFile"
        ]
    },
    {
        "category": "Calendar",
        "keywords": [
            "CalendarContract", "Events", "Reminders", "Calendars", "eventTitle",
            "eventDescription", "eventStart", "eventEnd", "eventLocation",
            "eventOrganizer", "alarm", "calendarAccount"
        ]
    },
    {
        "category": "Contacts",
        "keywords": [
            "ContactsContract", "getContentResolver", "contact_name", "contact_number",
            "phonebook", "callLog", "READ_CONTACTS", "WRITE_CONTACTS", "phoneState",
            "getLine1Number"
        ]
    },
    {
        "category": "App activity",
        "keywords": [
            "ActivityRecognition", "UsageStatsManager", "UsageEvents",
            "InAppSearch", "searchHistory", "installedApps", "PackageManager",
            "getInstalledPackages", "getInstalledApplications", "AppInteraction"
        ]
    },
    {
        "category": "Web browsing",
        "keywords": [
            "WebView", "CookieManager", "getCookie", "setCookie", "cache",
            "clearCache", "history", "browser", "URL", "http", "https"
        ]
    }
]


In [104]:
array_java_method = [
    {
        "category": "Device or other IDs",
        "methods": [
            "getDeviceId", "getImei", "getMeid", "getString", "getSerial",
            "getAddress", "getConnectionInfo", "getAdvertisingIdInfo", "getId",
            "getToken", "getPropertyByteArray", "getHardwareAddress", "getHostAddress",
            "getIpAddress"
        ]
    },
    {
        "category": "Personal info",
        "methods": [
            "getAccounts", "getAccountsByType", "getLine1Number",
            "getEmail", "getDisplayName", "getUid", "getPhoneNumber"
        ]
    },
    {
        "category": "Location",
        "methods": [
            "requestLocationUpdates", "getLastKnownLocation",
            "getLastLocation", "getFromLocation",
            "getLatitude", "getLongitude", "getAltitude"
        ]
    },
    {
        "category": "Health & fitness",
        "methods": [
            "subscribe", "add", "readData", "readRecords",
            "registerListener", "getType"
        ]
    },
    {
        "category": "App info & performance",
        "methods": [
            "log", "recordException", "getMemoryInfo",
            "getElapsedCpuTime", "getIntProperty",
            "totalMemory", "setDefaultUncaughtExceptionHandler"
        ]
    },
    {
        "category": "Messages",
        "methods": [
            "sendTextMessage", "sendMultipartTextMessage", "query", "send"
        ]
    },
    {
        "category": "Financial info",
        "methods": [
            "create", "loadPaymentData", "queryPurchases", "queryPurchaseHistoryAsync"
        ]
    },
    {
        "category": "Photos or video",
        "methods": [
            "open", "takePicture", "recordVideo",
            "ACTION_IMAGE_CAPTURE", "ACTION_VIDEO_CAPTURE"
        ]
    },
    {
        "category": "Audio files",
        "methods": [
            "start", "setAudioSource", "startRecording"
        ]
    },
    {
        "category": "Files and docs",
        "methods": [
            "read", "write", "openFileInput", "openFileOutput",
            "getExternalStorageDirectory"
        ]
    },
    {
        "category": "Calendar",
        "methods": [
            "insert", "query", "update"
        ]
    },
    {
        "category": "Contacts",
        "methods": [
            "query", "getLine1Number"
        ]
    },
    {
        "category": "App activity",
        "methods": [
            "queryUsageStats", "queryEvents", "requestActivityUpdates",
            "getInstalledPackages", "getInstalledApplications"
        ]
    },
    {
        "category": "Web browsing",
        "methods": [
            "getSettings", "setJavaScriptEnabled", "getCookie",
            "setCookie", "clearCache", "shouldOverrideUrlLoading"
        ]
    }
]


In [105]:
array_java_object_method = [
    {
        "category": "General set & assignment",
        "methods": [
            "set", "setValue", "setText", "setData", "setContent",
            "setExtra", "setExtras", "setProperty", "setAttribute",
            "setUser", "setUserId", "setEmail", "setPassword",
            "setLatitude", "setLongitude", "setLocation",
            "setDeviceId", "setPhoneNumber", "setAddress"
        ]
    },
    {
        "category": "Put / add to collections or containers",
        "methods": [
            "put", "putString", "putInt", "putLong", "putBoolean",
            "putExtra", "putExtras", "putParcelable", "putSerializable",
            "putAll", "add", "addAll", "append", "insert"
        ]
    },
    {
        "category": "Save / write to storage",
        "methods": [
            "save", "saveFile", "saveToFile", "saveData", "saveUser",
            "write", "writeToFile", "writeObject", "writeBytes", "writeValue",
            "store", "storeData", "persist", "commit", "apply",  # SharedPreferences
            "flush", "sync"
        ]
    },
    {
        "category": "Database operations",
        "methods": [
            "insert", "update", "replace", "execSQL", "executeUpdate",
            "saveOrUpdate", "writeToDatabase", "commitTransaction"
        ]
    },
    {
        "category": "Intent / Bundle / SharedPreferences",
        "methods": [
            "putExtra", "putExtras", "putString", "putInt", "putBoolean",
            "putLong", "putFloat", "putDouble", "putCharSequence",
            "edit", "commit", "apply"
        ]
    },
    {
        "category": "JSON / XML serialization",
        "methods": [
            "put", "accumulate", "append", "toJson", "toJsonString",
            "writeValueAsString", "serialize", "serializeObject",
            "setAttribute", "addProperty"
        ]
    },
    {
        "category": "File I/O stream writing",
        "methods": [
            "write", "writeBytes", "writeChars", "writeUTF",
            "flush", "close", "outputStream", "saveFile"
        ]
    },
    {
        "category": "Logging / debugging output",
        "methods": [
            "Log.d", "Log.i", "Log.w", "Log.e", "print", "println", "printf"
        ]
    }
]


In [106]:
def _read_text(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception:
        return ""

def _walk_java_files(root_dir):
    for dirpath, _, files in os.walk(root_dir):
        for fn in files:
            if fn.endswith(".java"):
                yield os.path.join(dirpath, fn)

def _flatten_terms(items, key):
    terms = []
    for item in items:
        terms.extend(item.get(key, []))
    # so khớp không phân biệt hoa/thường
    return list({t.lower() for t in terms if isinstance(t, str) and t.strip()})

def _ensure_dir(p):
    if not os.path.exists(p):
        os.makedirs(p)

def _unique_copy_path(dest_dir, filename):
    """
    Nếu filename đã tồn tại trong dest_dir, tạo tên mới:
    base-duplicate-name-{i}.ext
    """
    base, ext = os.path.splitext(filename)
    candidate = os.path.join(dest_dir, filename)
    i = 1
    while os.path.exists(candidate):
        candidate = os.path.join(dest_dir, f"{base}-duplicate-name-{i}{ext}")
        i += 1
    return candidate

def scan_and_copy_sensitive_source(
    source_dir,
    dest_dir,
    array_sensitive_keyword,
    array_java_method,
    array_java_object_method
):
    """
    Stage 1: quét toàn bộ source bằng array_sensitive_keyword -> result_1
    Stage 2: quét CHỈ trên result_1 bằng array_java_method -> result_2
    Stage 3: quét CHỈ trên result_2 bằng array_java_object_method -> result_3
    Copy tất cả file trong result_3 sang dest_dir (xử lý trùng tên).
    Trả về: danh sách đường dẫn file ĐÍCH đã copy (tương ứng result_3).
    """
    # ---- Stage 1: keywords ----
    keywords = _flatten_terms(array_sensitive_keyword, "keywords")
    result_1 = set()
    for path in _walk_java_files(source_dir):
        text = _read_text(path)
        if text and any(k in text.lower() for k in keywords):
            result_1.add(path)

    # ---- Stage 2: methods (getter/access) trên result_1 ----
    methods = _flatten_terms(array_java_method, "methods")
    result_2 = []
    for path in sorted(result_1):
        text = _read_text(path)
        if text and any(m in text.lower() for m in methods):
            result_2.append(path)

    # ---- Stage 3: object/sink methods trên result_2 ----
    obj_methods = _flatten_terms(array_java_object_method, "methods")
    result_3 = []
    for path in sorted(result_2):
        text = _read_text(path)
        if text and any(mm in text.lower() for mm in obj_methods):
            result_3.append(path)

    # ---- Copy chỉ result_3 ----
    _ensure_dir(dest_dir)
    copied_paths = []
    for src in result_3:
        filename = os.path.basename(src)
        dst = _unique_copy_path(dest_dir, filename)
        shutil.copy2(src, dst)
        copied_paths.append(dst)

    # --- log ngắn ---
    print(f"Stage-1 files (keywords): {len(result_1)}")
    print(f"Stage-2 files (methods within Stage-1): {len(result_2)}")
    print(f"Stage-3 files (object/sink methods within Stage-2): {len(result_3)}")
    print(f"Copied to '{dest_dir}': {len(copied_paths)} files")

    # Chỉ return result_3 đã copy
    return copied_paths

In [107]:
level_1_directory_arr = get_all_subdirectories_under_one_level(root_directory_source_code)
# print(len(level_1_directory_arr))

In [108]:
for i in range(2,len(level_1_directory_arr)):
    level_1_directory_path = level_1_directory_arr[i]
    package_name = get_app_package_name(level_1_directory_path)
    extract_code_directory_path = root_directory_extract_code+"\\"+package_name
    create_directory_if_not_exist(extract_code_directory_path)
    print("----------------------------------- Loop-"+str(i)+"-----------------------------------")
    print("Package name: ",package_name)
    copied = scan_and_copy_sensitive_source(
        source_dir=level_1_directory_path,
        dest_dir=extract_code_directory_path,
        array_sensitive_keyword=array_sensitive_keyword,
        array_java_method=array_java_method,
        array_java_object_method=array_java_object_method
    )
    # break   

----------------------------------- Loop-2-----------------------------------
Package name:  ch.publisheria.bring.apk
Stage-1 files (keywords): 12528
Stage-2 files (methods within Stage-1): 7303
Stage-3 files (object/sink methods within Stage-2): 6171
Copied to 'E:\wearable-2-sharing-compliance-code-extraction\ch.publisheria.bring.apk': 6171 files
----------------------------------- Loop-3-----------------------------------
Package name:  com.albuquerquedesign.adanalog013.apk
Stage-1 files (keywords): 2266
Stage-2 files (methods within Stage-1): 1305
Stage-3 files (object/sink methods within Stage-2): 1200
Copied to 'E:\wearable-2-sharing-compliance-code-extraction\com.albuquerquedesign.adanalog013.apk': 1200 files
----------------------------------- Loop-4-----------------------------------
Package name:  com.anghami.apk
Stage-1 files (keywords): 10547
Stage-2 files (methods within Stage-1): 6644
Stage-3 files (object/sink methods within Stage-2): 5891
Copied to 'E:\wearable-2-sharing